In [1]:
%pip install -U langchain

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers pandas datasets

Note: you may need to restart the kernel to use updated packages.


In [3]:
import langchain
import pandas
import faiss
import sentence_transformers
import transformers
import datasets

print("All packages installed successfully")

All packages installed successfully


In [4]:
from datasets import load_dataset
dataset = load_dataset("batuhanmtl/job-skill-set")

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['job_id', 'category', 'job_title', 'job_description', 'job_skill_set'],
        num_rows: 1167
    })
})

In [6]:
df = dataset["train"].to_pandas()

In [7]:
df.sample(5)

,job_id,category,job_title,job_description,job_skill_set
951,3905305873,BUSINESS-DEVELOPMENT,Business Development Coordinator,"THOMPSON COBURN, A LARGE FULL-SERVICE LAW FIRM...","['communication', 'marketing', 'proposal creat..."
740,3902828623,FINANCE,Fractional Finance Consultant,VACO HAS PARTNERED WITH A LOCAL AIRCRAFT COMPA...,"['finance', 'accounting', 'financial oversight..."
72,3898176212,HR,Vice President Human Resources,VICE PRESIDENT OF HUMAN RESOURCESLOCATION: PRI...,"['Human Resources Management', 'Conflict Resol..."
802,3900087591,FINANCE,IT Finance Operations Manager,"WHEN YOU JOIN THE GROWING BILH TEAM, YOU'RE NO...","['financial analysis', 'budget management', 'd..."
369,3904729333,INFORMATION-TECHNOLOGY,Information Technology Program Manager,AKKODIS IS PARTNERING WITH A LEADING COMPANY A...,"['Communication', 'Leadership', 'Proactivity',..."


In [8]:
from langchain_core.documents import Document

documents = []

for _, row in df.iterrows():
    doc = Document(
        page_content=row["job_description"],
        metadata={
            "job_id": row["job_id"],
            "category": row["category"],
            "job_title": row["job_title"],
            "job_skill_set": row["job_skill_set"]
        }
    )
    documents.append(doc)

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size= 800, chunk_overlap = 100)
chunks = splitter.split_documents(documents)

In [10]:
len(chunks)

8122

In [11]:
print(chunks[42].page_content)

AD&D, FSA, 401(K) RETIREMENT PLAN, AND AFLAC BENEFIT OFFERINGSB. REVIEW AND ENSURE ICS BENEFITS PROGRAMS ARE COMPETITIVE, COST‐EFFECTIVE, AND SATISFY EMPLOYEE NEEDS.C. PARTNER WITH INSURANCE BROKERS AND CARRIERS TO RESOLVE BENEFIT ISSUES.D. MANAGE WEB‐BASED ENROLLMENTS DIRECTLY ON THE CARRIER PORTAL AND IN THE IN‐HOUSE PAYROLL SYSTEM TOENSURE TIMELY MEMBERSHIP ID CARDS ARE PROCESSED, AND CONTRIBUTION DEDUCTIONS ARE SET UP IN PAYROLL.E. MANAGE OPEN ENROLLMENT AND THE DAILY ADMINISTRATION OF COMPANY BENEFITS, AND RETIREMENT PLANS AND UTILIZE POSITIVE PROFESSIONAL RELATIONSHIPS WITH BROKERS, CARRIERS, THIRD‐PARTY ADMINISTRATORS, AND COLLEAGUES TO MAXIMIZE A POSITIVE EMPLOYEE EXPERIENCE.F. LEAD COMPANY‐WIDE BENEFIT PLAN/PROGRAM RESEARCH, ANALYSIS, AND PROJECTS. COLLABORATE WITH THELEADERSHIP


In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

In [13]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [14]:
vector =  embeddings.embed_query(chunks[0].page_content)


In [15]:
print(len(vector))

384


In [16]:
from langchain_community.vectorstores import FAISS
vector_store = FAISS.from_documents(chunks, embeddings)

C:\Users\rites\AppData\Local\Temp\ipykernel_17740\3484710893.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [17]:
vector_store.index_to_docstore_id

{0: '87cac2c5-285a-471e-8197-37545970fe6e',
 1: '05767537-ce1d-4525-a6cb-b1a682d3af21',
 2: '87b1a997-cfe9-424f-b391-ef87d153334d',
 3: 'd49421f1-22dc-4a5a-bdf9-aef358fc50c2',
 4: '7d2923a9-bee3-4fa5-838d-738924d7671b',
 5: '7a84b185-7fa8-404c-8d6d-0e1552ee2e51',
 6: 'e8115a94-1515-4ae3-b073-6919bfacc58c',
 7: 'f6aee5eb-b361-4e80-9c7c-de0e4783a148',
 8: 'e449c2d3-92fc-416f-b4e7-aa090ad72bb6',
 9: '2f73a878-a5fb-4afe-b6c1-955397258bb2',
 10: 'ea3d00e9-c341-43a1-92ec-5fdefaa69b57',
 11: '8b1025cf-55ac-4eac-8cf7-4102447018e2',
 12: '2b342cf5-32e8-462b-99b2-2beab7167ed8',
 13: 'd4b83229-9f53-4ab6-ab8e-7affaf07e8e3',
 14: 'c74b4422-7ec5-4f71-af60-8e53b44d9514',
 15: '1c54610b-91f5-4d44-bc4a-8cc95e24052e',
 16: '40c06ee0-1556-432f-8a5d-63fa5644622a',
 17: 'ee57f9a3-c402-4a6d-a453-5ca430929fa1',
 18: 'ca9403a8-dd74-464c-8be0-e2a33541c61a',
 19: 'eb23b26e-a1e3-4b29-a91c-3810d7b4d454',
 20: 'a4395807-8279-4846-9b37-29cc5908d156',
 21: 'f173140f-90a3-428e-a021-d9a45f92705d',
 22: '3834115c-6049-

In [18]:
vector_store.get_by_ids(['240bb5af-f601-4bad-bd36-9af54bbd06c5'])

[]

In [19]:
# RAG pipeline starts

retriever = vector_store.as_retriever(search_type= 'similarity' , search_kwargs={"k":4})

In [20]:
results = retriever.invoke("job skill set for finance")
print(results)

[Document(id='d229e54d-71f6-496e-8bf4-6a9ea7602ce3', metadata={'job_id': 3905350188, 'category': 'FINANCE', 'job_title': 'Strategic Finance Manager', 'job_skill_set': "['financial analysis', 'budget management', 'financial forecasting', 'data analysis', 'financial software', 'GAAP', 'communication', 'analytical skills', 'interpersonal skills', 'collaboration', 'problem solving', 'innovation', 'financial planning', 'financial modeling', 'advanced Excel', 'eCommerce', 'marketplace experience']"}, page_content='QUALIFICATIONS:BACHELOR’S DEGREE IN FINANCE, ACCOUNTING, ECONOMICS, OR RELATED FIELD.HAS 6+ YEARS OF EXPERIENCE IN STRATEGIC FINANCE, FP&A, BIG 4 ACCOUNTING, INVESTMENT BANKING, OR CORPORATE FINANCE.PRIOR EXPERIENCE IN HIGH-GROWTH OR STARTUP ENVIRONMENTS IS REQUIRED.ADVANCED FINANCIAL MODELING TECHNIQUES AND ABILITIES WITH A DEEP UNDERSTANDING OF BEST FINANCIAL MODELING PRACTICES.EXCEPTIONAL ANALYTICAL SKILLS AND THE ABILITY TO SYNTHESIZE COMPLEX FINANCIAL INFORMATION INTO CLEAR, C

In [21]:
from langchain_core.prompts import ChatPromptTemplate
chat_template = ChatPromptTemplate([
  ('system' , 'You are a helpful job information expert.'),
   ('human' , 'Context:\n {context}\n\n Question:\n{question}')

])

In [25]:
context = "\n\n".join(
    doc.page_content for doc in results)

In [26]:
question = "job skill set for information technology"
results = retriever.invoke(question)

In [27]:
prompt = chat_template.invoke({'question':question , 'context':context})

In [28]:
prompt

ChatPromptValue(messages=[SystemMessage(content='You are a helpful job information expert.', additional_kwargs={}, response_metadata={}), HumanMessage(content="Context:\n QUALIFICATIONREQUIREMENTS:MUST BE ABLE TO PERFORM PC-BASED HARDWARE, SOFTWARE, AND NETWORKING DUTIES SATISFACTORILY. THE REQUIREMENTS LISTED BELOW ARE REPRESENTATIVE OF THE KNOWLEDGE, SKILL, AND ABILITY REQUIRED. EDUCATIONANDEXPERIENCE: TECHNICAL KNOWLEDGE OF INFORMATION TECHNOLOGY (IT) SUBJECTS, INCLUDING A BASIC UNDERSTANDING OF PERSONAL COMPUTER (PC) HARDWARE, MS OFFICE APPLICATIONS INCLUDING WORD, POWERPOINT, EXCEL, AND OUTLOOK. EXPERIENCE WITH LAN INFRASTRUCTURE, VMWARE ESXI, MICROSOFT SERVERS, PRINTERS, COPIERS, PHONE & VOICEMAIL SYSTEMS, AND SERVER BACKUP SOFTWARE ARE PREFERRED.ASSOCIATE DEGREE OR HIGHER IN AN IT FIELD IS ALSO PREFERRED.\n\nATTRACT STUDENTS TO A CAREER IN COMPUTER INFORMATION TECHNOLOGY.PARTICIPATE AS A COOPERATIVE TEAM MEMBER OF THE COMPUTER INFORMATION TECHNOLOGY FACULTY TO STRENGTHEN THE PRO

In [29]:
%pip install -U langchain-huggingface huggingface_hub

  Using cached huggingface_hub-2.2.0-py3-none-any.whl.metadata (16 kB)
Note: you may need to restart the kernel to use updated packages.


In [31]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [32]:
%pip install -U langchain-google-genai

Note: you may need to restart the kernel to use updated packages.


In [33]:
from google import genai

client = genai.Client()


In [34]:
from langchain_google_genai import ChatGoogleGenerativeAI
model= ChatGoogleGenerativeAI(model = 'gemini-3.5-flash-lite')

In [35]:
response = model.invoke(prompt)

print(response.content)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'Based on the provided context, the required and preferred **Information Technology (IT)** skill set spans hardware, software, networking, administration, and support. Here is a breakdown of the specific IT skills and technical requirements mentioned:\n\n### **1. Hardware & Physical IT Skills**\n* Perform PC-based hardware duties satisfactorily.\n* Basic understanding of personal computer (PC) hardware.\n* Experience with hardware troubleshooting.\n* Setup, maintenance, and troubleshooting of printers, copiers, phone & voicemail systems, and media equipment.\n* Ability to lift and carry up to 50 pounds (for moving equipment, tables, and chairs).\n\n### **2. Software & Operating Systems**\n* Microsoft Environments, including Windows Operating Systems and Microsoft Servers.\n* MS Office Applications (Word, PowerPoint, Excel, and Outlook) and Office Suite.\n* Google Workspace.\n* Oracle.\n* Server backup software.\n* VMware ESXi (preferred).\n* Software troublesh

In [36]:
def format_docs(docs):
    return "\n\n".join(
        doc.page_content for doc in docs
    )

In [37]:
from langchain_core.runnables import RunnablePassthrough


In [38]:
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | chat_template
    | model
)

In [39]:
response = rag_chain.invoke(
    "job skill set for information technology"
)


In [40]:
print(response.content)

[{'type': 'text', 'text': 'Based on the provided context, the required and preferred **Information Technology (IT) skill set** spans several roles (IT support, networking, hardware/software, administration, and contract management). \n\nHere is the comprehensive IT skill set extracted from the text:\n\n### **Technical Skills & Knowledge**\n* **Hardware & Software:** \n  * PC-based hardware and software duties, installation, and troubleshooting.\n  * Experience with printers, copiers, phone & voicemail systems, and media equipment.\n  * Microsoft Environments: Windows Operating Systems, MS Office Applications (Word, PowerPoint, Excel, Outlook), and Microsoft Servers.\n  * Server backup software.\n* **Networking & Infrastructure:** \n  * LAN Infrastructure and VMware ESXi.\n  * Networking concepts, including TCP/IP, DNS, DHCP, and VPN.\n* **Systems & Enterprise Tools:** \n  * Active Directory.\n  * Specialized software such as Oracle and Google Workspace (preferred).\n* **Research & Tech

In [41]:
%pip install fastapi uvicorn

Note: you may need to restart the kernel to use updated packages.


In [42]:
vector_store.save_local("faiss_index")

print("FAISS index saved successfully!")

FAISS index saved successfully!
